# Empirical Risk Premia Engine v0.2.5 — Rates: full v0.8 forecasting layer

**Recovered research pipeline:** Federal Reserve Kim–Wright fitted zero curve → vol-standardized PCA (risk exposures) → exact funded factor-mimicking portfolios → *carry and curve repricing* → **Fama–Bliss / Cochrane–Piazzesi-style empirical bond-premia forecasts** → 3/6/12M targets → mean-confidence rotation, monthly execution, transaction costs, OOS prediction & strategy evaluation.

Preserves five historical predictor families **BASE_STATE, FB, FB_STATE, CP, CP_STATE** and **UNCONDITIONAL / CARRY_ONLY** benchmarks. The v0.8 predictor, signal and backtest functions are copied unchanged into `erp.rates_research` and covered by source-parity tests. The funded Rates adapter retains v0.2.3 cash/no-stale-funding safeguards.

**Crucial:** *Risk PCA is NOT treated as proof that Level/Slope/Curvature are independently priced shocks.* Factor portfolio returns are forecast targets. FB/CP predictor variables are not additional traded risk factors.

**Data:** No proxy Treasury par yields, no imputed macro values. Historical revised observations are **not genuine historical publication vintages**.

In [1]:
# Portable ERP v0.2.5 — actual Python modules packaged inside this notebook.
import io, zipfile, base64, sys, subprocess, importlib.util
from pathlib import Path
runtime_root=Path.cwd()/'erp_v025_notebook_runtime'
(runtime_root/'src').mkdir(parents=True,exist_ok=True)
_PAYLOAD='...'
with zipfile.ZipFile(io.BytesIO(base64.b64decode(_PAYLOAD))) as archive:
    archive.extractall(runtime_root/'src')
for mod,pkg in {'numpy':'numpy','pandas':'pandas','scipy':'scipy','statsmodels':'statsmodels','requests':'requests','openpyxl':'openpyxl'}.items():
    if importlib.util.find_spec(mod) is None:subprocess.check_call([sys.executable,'-m','pip','install','-q',pkg])
sys.path.insert(0,str(runtime_root/'src'))
for _key in list(sys.modules):
    if _key=='erp' or _key.startswith('erp.'): del sys.modules[_key]
from erp import __version__, RatesData, RatesPremiaEngine, RatesEmpiricalBondPremiaEngine, load_macro_panel, load_macro_series_files, export_snapshot
import pandas as pd
from IPython.display import display
print('ERP:',__version__,'| Rates v0.8 forecasting layer + v0.2.3 funded adapter')


ERP: 0.2.5 | Rates v0.8 forecasting layer + v0.2.3 funded adapter


## Configuration and external data

The curve loads from one [Federal Reserve Board Kim–Wright CSV](https://www.federalreserve.gov/data/yield-curve-tables/feds200533.csv) with fallback to a locally uploaded file. Funding is the original **FEDFUNDS** series, now with local/mirror options established in v0.2.3. To avoid another remote provider failure, upload cached originals when possible.

**All five model families now load macro data automatically where available.** The loader requests the original FRED `CPIAUCSL`, `INDPRO`, `T10YIE` series as one export (CSV or ZIP), caches it and validates units/coverage. FRED downloads may still be blocked in Colab. If this happens, the notebook offers a **single-file upload**, accepting the [original combined FRED CSV/ZIP](https://fred.stlouisfed.org/graph/fredgraph.csv?id=CPIAUCSL%2CINDPRO%2CT10YIE), or three raw original FRED CSVs. No proxy factors, no forward-filled observations, and no synthetic data. Historical revised observations are not real-time vintages. If you intentionally want FB/CP alone, set `REQUIRE_FULL_GRID=False` and pass `MACRO_SOURCE='none'` only in the model cell.

In [2]:
AS_OF='2026-10-31'
CACHE_DIR=Path('erp_data_cache')
OUT_DIR=Path('erp_v025_output')
CURVE_SOURCE='board'
BOARD_CSV_PATH=None         # e.g. '/content/feds200533.csv'
FUNDING_SOURCE='mirror'    # v0.2.3 fallback; 'local' with CSV below for no network
FEDFUNDS_CSV_PATH=None      # e.g. '/content/FEDFUNDS.csv'
MACRO_CSV_PATH=None         # optional: '/content/macro_original.zip' or .csv
MACRO_SOURCE='auto'        # auto = local cache then ONE FRED combined download; local = file only
AUTO_UPLOAD_ON_FAILURE=True # Colab prompts to upload original CSV/ZIP if FRED is unreachable
REQUIRE_FULL_GRID=True    # no silent skip: request all 5 families, or fail with instructions
HORIZONS=(3,6,12)
MIN_TRAIN=60
RIDGE=1.0
COST_BPS=5.0
COST_GRID=(1.0,5.0,10.0)


## 1 — Funded Rates decomposition (preserved v0.2.3)

`as_of` is a strict observation cutoff. The output `snapshot.factor_premia` is the **unconditional mean of funded factor portfolio returns**; the new conditional estimates are in the separate results object.

In [3]:
data=RatesData.from_fred(
    cache_dir=CACHE_DIR,curve_source=CURVE_SOURCE,
    local_curve_path=BOARD_CSV_PATH,
    local_fedfunds_path=FEDFUNDS_CSV_PATH,
    funding_source=FUNDING_SOURCE,
)
rates=RatesPremiaEngine(data)
snapshot=rates.fit(as_of=AS_OF,pca_train_end='2002-12-31')
print('Observation cutoff:',snapshot.as_of)
print('Funded factor portfolio mean returns (monthly decimals):')
display(snapshot.factor_premia.to_frame('mean_monthly'))
print('PCA explained variance (risk, NOT risk premium):')
display(rates.pca.explained_variance_ratio.to_frame('fraction'))


Treasury curve: loading one official Kim-Wright Board CSV (10 tenors).
Funding: verified FEDFUNDS mirror (868 months through 2026-10-31).
Kim-Wright curve ready: 443 months, 1990-01-31 to 2026-11-30
FEDFUNDS ready: 868 months, latest 2026-10-31
Observation cutoff: 2026-10-31 00:00:00
Funded factor portfolio mean returns (monthly decimals):


,mean_monthly
Level,-0.000664
Slope,-0.000028
Curvature,0.000004


PCA explained variance (risk, NOT risk premium):


,fraction
Level,0.929265
Slope,0.065650
Curvature,0.005069


## 2 — Original v0.8 empirical bond-premia research grid

Forecast targets use actual **H-month excess bond returns**, realized cash over H months, and *funded carry known at forecast date*; curve repricing is the residual. Training labels for a forecast made at `t` are from origins `s <= t-H`, never incomplete overlapping H-month outcomes. Forecasts target 3/6/12 months while **trades are monthly**, exactly as in v0.8.

In [4]:
# The STATE families require genuine (not proxy) CPIAUCSL, INDPRO and T10YIE.
# When FRED is blocked in Colab, select the original combined macro CSV/ZIP
# downloaded from the FRED link above, or the three individual FRED exports.
macro_raw=None
if MACRO_SOURCE != 'none':
    try:
        macro_raw=load_macro_panel(cache_dir=CACHE_DIR,source=MACRO_SOURCE,
                                   path=MACRO_CSV_PATH,as_of=AS_OF)
    except (ConnectionError,FileNotFoundError) as ex:
        print('Automatic macro loading unavailable:',str(ex))
        if not AUTO_UPLOAD_ON_FAILURE:
            raise
        try:
            from google.colab import files as colab_files
        except ImportError:
            raise RuntimeError('Outside Colab: set MACRO_CSV_PATH to original FRED combined CSV/ZIP.') from ex
        print('Upload original FRED combined CSV/ZIP, or all three individual raw CSVs.')
        incoming=colab_files.upload()
        if not incoming:
            raise RuntimeError('No original macro file uploaded; cannot execute full five-family grid.') from ex
        if len(incoming)==1:
            only=next(iter(incoming))
            macro_raw=load_macro_panel(cache_dir=CACHE_DIR,source='local',path=only,as_of=AS_OF)
        else:
            mapping={sid: next((n for n in incoming if n.upper().startswith(sid)), None)
                     for sid in ['CPIAUCSL','INDPRO','T10YIE']}
            if None in mapping.values():
                raise ValueError('Upload one combined original macro CSV/ZIP, or separately named CPIAUCSL.csv, INDPRO.csv, T10YIE.csv.')
            macro_raw=load_macro_series_files(mapping,as_of=AS_OF)
if macro_raw is not None:
    print('Original macro panel coverage / last observed valid month:')
    display(pd.DataFrame({'first':macro_raw.apply(lambda v:v.first_valid_index()),
                          'last':macro_raw.apply(lambda v:v.last_valid_index()),
                          'count':macro_raw.count()}))
research=RatesEmpiricalBondPremiaEngine(rates).fit(
    snapshot=snapshot,macro_raw=macro_raw,horizons=HORIZONS,
    min_train=MIN_TRAIN,ridge=RIDGE,cost_bps=COST_BPS,
    cost_sensitivity=COST_GRID,require_full_grid=REQUIRE_FULL_GRID,
)
print('Models requested:',research.metadata['models_requested'])
print('Models executed:',research.metadata['models_run'])
print('Models skipped for missing original macro data:',research.metadata['models_skipped_for_missing_macro'])
print('Forecast counts (date × factor):',[(str(k),len(v)) for k,v in research.forecasts.items()])


Automatic macro loading unavailable: CPIAUCSL/INDPRO/T10YIE could not be downloaded from FRED. Download the original CSV or ZIP using your browser and specify MACRO_CSV_PATH. No alternative macro variables or forward-filled values will be substituted. Underlying error: HTTPSConnectionPool(host='fred.stlouisfed.org', port=443): Read timed out. (read timeout=20)
Upload original FRED combined CSV/ZIP, or all three individual raw CSVs.


<IPython.core.display.HTML object>

Saving fredgraph (4).zip to fredgraph (4).zip
Macro source: local fredgraph (4).zip; CPIAUCSL, INDPRO, T10YIE in original units.
Original macro panel coverage / last observed valid month:


,first,last,count
CPI,1947-01-31,2026-10-31,957
INDPRO,1919-01-31,2026-10-31,1294
BREAKEVEN10Y,2003-01-31,2026-10-31,286


/content/erp_v025_notebook_runtime/src/erp/rates_research.py:1657: FutureWarning: The default fill_method='pad' in Series.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  state['inflation_yoy']=m['CPI'].pct_change(12).shift(1)


Models requested: ['BASE_STATE', 'FB', 'FB_STATE', 'CP', 'CP_STATE']
Models executed: ['BASE_STATE', 'FB', 'FB_STATE', 'CP', 'CP_STATE']
Models skipped for missing original macro data: []
Forecast counts (date × factor): [("(3, 'BASE_STATE')", 558), ("(3, 'FB')", 666), ("(3, 'FB_STATE')", 558), ("(3, 'CP')", 666), ("(3, 'CP_STATE')", 558), ("(6, 'BASE_STATE')", 540), ("(6, 'FB')", 648), ("(6, 'FB_STATE')", 540), ("(6, 'CP')", 648), ("(6, 'CP_STATE')", 540), ("(12, 'BASE_STATE')", 504), ("(12, 'FB')", 612), ("(12, 'FB_STATE')", 504), ("(12, 'CP')", 612), ("(12, 'CP_STATE')", 504)]


## 3 — Classic Fama–Bliss 12M diagnostic & predictive diagnostics

Horizon-specific forecast diagnostics compare the model's **curve-repricing** forecasts to expanding unconditional means. `OOS_R2_vs_uncond` can be negative even when a model has nonzero correlation. Do not compare models on different samples without re-aligning; a fully matched common-sample horse race belongs to v0.9.

In [5]:
if not research.fb_12m_diagnostics.empty:
    print('Fama–Bliss canonical 12M diagnostics (HAC t):')
    display(research.fb_12m_diagnostics)
print('OOS predictive diagnostics, by horizon and model (factor averages):')
if len(research.predictive_diagnostics):
    display(research.predictive_diagnostics.groupby(['horizon','model'])[['corr','OOS_R2_vs_uncond','MAE','n']].mean().round(4))


Fama–Bliss canonical 12M diagnostics (HAC t):


,beta,t_HAC,R2,n
ZC2Y,0.618415,1.526264,0.035559,430.0
ZC3Y,0.688331,1.806602,0.045100,430.0
ZC4Y,0.776962,2.098723,0.055791,430.0
ZC5Y,0.883242,2.381609,0.067081,430.0


OOS predictive diagnostics, by horizon and model (factor averages):


corr  OOS_R2_vs_uncond     MAE      n
horizon model                                              
3       BASE_STATE  0.3153           -0.1205  0.0051  186.0
        CP          0.0289           -0.0285  0.0043  222.0
        CP_STATE    0.2669           -0.2006  0.0050  186.0
        FB          0.2204            0.0766  0.0043  222.0
        FB_STATE    0.3058           -0.1513  0.0054  186.0
6       BASE_STATE  0.3557           -0.3694  0.0089  180.0
        CP         -0.0532           -0.0911  0.0065  216.0
        CP_STATE    0.3023           -0.5068  0.0089  180.0
        FB          0.2890            0.1138  0.0063  216.0
        FB_STATE    0.3205           -0.3879  0.0091  180.0
12      BASE_STATE  0.4296           -0.3745  0.0135  168.0
        CP          0.1248           -0.0847  0.0096  204.0
        CP_STATE    0.3644           -0.6705  0.0156  168.0
        FB          0.4144            0.1348  0.0096  204.0
        FB_STATE    0.1827           -0.6234  0.0138  168.0

## 4 — Rotation with funded exposures, monthly execution and 5-bp cost

Exactly the v0.8 mean-confidence signal: estimated mean uncertainty is not conflated with next realized-return volatility. Factor allocations are volatility-scaled and capped; underlying maturity gross exposure is capped separately. Results contain cash return, factor overlay return and turnover costs.

In [6]:
print('Performance matrix (CAGR, Sharpe, MaxDD):')
if len(research.strategy_results):
    display(research.strategy_results[['CAGR','Sharpe','MaxDD','AvgMonthlyTurnover']].round(4))
print('Conditional vs horizon-matched unconditional strategy, classic and HAC t:')
display(research.conditional_value_add.round(4))
print('Cost sensitivity (gross exposure and turnover accounted):')
if len(research.cost_sensitivity):
    display(research.cost_sensitivity.pivot_table(index=['horizon','model'],columns='cost_bps',values='Sharpe').round(3))


Performance matrix (CAGR, Sharpe, MaxDD):


CAGR  Sharpe   MaxDD  AvgMonthlyTurnover
horizon model                                                    
3.0     BASE_STATE     0.0177  0.2010 -0.0128              0.1091
        CARRY_ONLY     0.0180  0.4741 -0.0169              0.0853
        CP             0.0170  0.2976 -0.0214              0.0940
        CP_STATE       0.0171  0.1716 -0.0090              0.1145
        FB             0.0179  0.2840 -0.0229              0.0900
        FB_STATE       0.0152 -0.0602 -0.0111              0.1204
        UNCONDITIONAL  0.0182  0.4549 -0.0172              0.0805
6.0     BASE_STATE     0.0164  0.0702 -0.0253              0.1041
        CARRY_ONLY     0.0181  0.5453 -0.0170              0.0807
        CP             0.0161  0.2497 -0.0252              0.0988
        CP_STATE       0.0145 -0.1880 -0.0217              0.1122
        FB             0.0174  0.3030 -0.0216              0.0914
        FB_STATE       0.0141 -0.1707 -0.0280              0.1140
        UNCONDITIONAL  0.0180  0.5078 -0.0178              0.0737
12.0    BASE_STATE     0.0175  0.2882 -0.0147              0.0686
        CARRY_ONLY     0.0156  0.3758 -0.0180              0.0732
        CP             0.0144  0.1847 -0.0146              0.0842
        CP_STATE       0.0164  0.1196 -0.0109              0.0662
        FB             0.0167  0.3360 -0.0169              0.0830
        FB_STATE       0.0145 -0.1304 -0.0155              0.0949
        UNCONDITIONAL  0.0157  0.3748 -0.0181              0.0653

Conditional vs horizon-matched unconditional strategy, classic and HAC t:


annualized_mean_spread  spread_t_stat  spread_t_hac  \
horizon model                                                             
3       BASE_STATE                  0.0005         0.2268        0.2200   
        FB                         -0.0002        -0.1578       -0.1687   
        FB_STATE                   -0.0019        -0.7421       -0.7850   
        CP                         -0.0011        -0.5376       -0.6617   
        CP_STATE                   -0.0001        -0.0504       -0.0463   
6       BASE_STATE                 -0.0002        -0.1122       -0.1020   
        FB                         -0.0005        -0.3386       -0.3620   
        FB_STATE                   -0.0024        -1.1777       -1.2307   
        CP                         -0.0019        -0.9907       -1.1638   
        CP_STATE                   -0.0020        -1.3865       -1.8856   
12      BASE_STATE                  0.0017         0.9057        0.9197   
        FB                          0.0010         0.4885        0.5371   
        FB_STATE                   -0.0013        -0.5209       -0.6251   
        CP                         -0.0013        -0.5762       -0.5808   
        CP_STATE                    0.0005         0.2488        0.2589   

                    n_months  
horizon model                 
3       BASE_STATE     186.0  
        FB             222.0  
        FB_STATE       186.0  
        CP             222.0  
        CP_STATE       186.0  
6       BASE_STATE     180.0  
        FB             216.0  
        FB_STATE       180.0  
        CP             216.0  
        CP_STATE       180.0  
12      BASE_STATE     168.0  
        FB             204.0  
        FB_STATE       168.0  
        CP             204.0  
        CP_STATE       168.0

Cost sensitivity (gross exposure and turnover accounted):


cost_bps             1.0    5.0    10.0
horizon model                          
3       BASE_STATE  0.263  0.201  0.123
        CP          0.350  0.298  0.232
        CP_STATE    0.254  0.172  0.069
        FB          0.319  0.284  0.240
        FB_STATE   -0.009 -0.060 -0.125
6       BASE_STATE  0.135  0.070 -0.011
        CP          0.305  0.250  0.180
        CP_STATE   -0.111 -0.188 -0.283
        FB          0.341  0.303  0.255
        FB_STATE   -0.116 -0.171 -0.239
12      BASE_STATE  0.339  0.288  0.225
        CP          0.244  0.185  0.111
        CP_STATE    0.172  0.120  0.055
        FB          0.374  0.336  0.289
        FB_STATE   -0.075 -0.130 -0.199

## 5 — CP coefficients and current *historical* allocations

These allocation dates are **the last origins with realized H-month targets in the retrospective v0.8 protocol**, not live deployment recommendations for the latest curve date. This distinction is preserved intentionally during source parity; latest-date deployable signals can be developed separately with their own no-leakage tests.

In [7]:
display(research.cp_coefficients.tail(8))
display(research.latest_allocations.tail(14))


,horizon,model,intercept,f1,f2,f3,f4,f5
0,3,CP,0.016806,54.738427,-327.367886,693.094607,-617.697366,197.335391
1,3,CP_STATE,0.016806,54.738427,-327.367886,693.094607,-617.697366,197.335391
2,6,CP,0.101080,69.273902,-384.514629,730.705407,-551.038924,135.385644
3,6,CP_STATE,0.101080,69.273902,-384.514629,730.705407,-551.038924,135.385644
4,12,CP,0.824755,234.306412,-1145.985037,1689.619747,-607.039476,-174.480821
5,12,CP_STATE,0.824755,234.306412,-1145.985037,1689.619747,-607.039476,-174.480821


,horizon,model,formation_date,Level,Slope,Curvature
7,6,UNCONDITIONAL,2026-04-30,-0.194061,-0.182792,0.623147
8,6,CARRY_ONLY,2026-04-30,-0.241836,0.094892,0.663272
9,6,BASE_STATE,2026-04-30,-0.353846,-0.191026,0.455128
10,6,FB,2026-04-30,-0.411184,-0.144955,0.443861
11,6,FB_STATE,2026-04-30,-0.041832,0.406211,0.551957
12,6,CP,2026-04-30,-0.228396,-0.140822,0.630782
13,6,CP_STATE,2026-04-30,-0.166210,-0.162463,0.671327
14,12,UNCONDITIONAL,2025-10-31,0.063644,-0.588686,-0.347670
15,12,CARRY_ONLY,2025-10-31,0.047549,-0.551730,-0.400721
16,12,BASE_STATE,2025-10-31,-0.305410,-0.694590,0.000000


## 6 — Export common snapshot and full v0.8 research result

The core snapshot remains compatible with the downstream Alpha Slicer/Referee factor library; new CSV files give TAA access to historical predictions, factor targets, weights and realized strategy returns. No change is made to those downstream repositories.

In [8]:
root=OUT_DIR/'rates_v025'
snapshot_dir=export_snapshot(snapshot,root/'core_factor_snapshot')
rates.export_auxiliary(snapshot_dir)
research_dir=research.export(root/'empirical_bond_premia')
print('Core exported to:',snapshot_dir)
print('FB/CP research exported to:',research_dir)
print('Files:',len(list(research_dir.glob('*'))))


Core exported to: erp_v025_output/rates_v025/core_factor_snapshot
FB/CP research exported to: erp_v025_output/rates_v025/empirical_bond_premia
Files: 101


## Validation limits and next research release

v0.2.5 is a **source-parity port**, not an improvement claim. Same forecast families, original v0.8 windows, standardized ridge and CP stage, horizon target calendar, and cash/carry decomposition. Macro-based models need original CPI/INDPRO/T10YIE observations; no automatic model substitution. Original v0.8 notebook is bundled in the source repo for review. Regression tests include exact source correspondence of historical functions and synthetic five-family end-to-end integration. **Historical-data numerical parity on the exact original FRED vintage remains unverified** because that complete original data snapshot has not been supplied.

v0.9 research agenda: align common samples across all models, investigate CP collinearity and the pricing-vs-repricing target, robust HAC/bootstrap comparisons, and bring forecast origins to the latest date without using incomplete realized labels.